# CoSQA E5-base-v2 full benchmark

This notebook is the full-dataset entry point for AZH-514. It always runs the complete declared CosQA test-qrels query population and the complete CosQA corpus. It does not have a smoke default and will fail if the selected data is not complete.

The experiment evaluates `intfloat/e5-base-v2` with text-only `query: ` / `passage: ` inputs, exact Faiss retrieval, and the official COIR evaluator at `nDCG@10`. For a quick wiring check, use `e5_baseline_experiment.ipynb` instead.

Reference: [CoIR: A Comprehensive Benchmark for Code Information Retrieval Models](https://arxiv.org/pdf/2407.02883).

In [ ]:
# Install code-retrieval/requirements.txt once before running this notebook if needed.
# In Colab, uncomment the next line and restart the runtime if pip updates torch/numpy.
# %pip install -r ../requirements.txt

from pathlib import Path
import hashlib
import json
import os
import sys
import time

workspace_root = Path.cwd()
while workspace_root != workspace_root.parent and not (workspace_root / 'code-retrieval' / 'src').exists():
    workspace_root = workspace_root.parent
project_dir = workspace_root / 'code-retrieval'
if not (project_dir / 'src').exists():
    project_dir = Path.cwd()
    workspace_root = project_dir.parent

os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))

from e5_baseline import (
    BaselineConfig,
    E5Encoder,
    build_result,
    cache_paths,
    environment_metadata,
    evaluate_ndcg_at_10,
    expected_cache_metadata,
    load_cosqa,
    load_valid_embedding_cache,
    load_valid_json_cache,
    package_versions,
    rank_with_coir_exact,
    run_identity,
    save_embedding_cache,
    save_json_cache,
    select_run_data,
    set_seed,
    write_result_artifacts,
)

print('Project directory:', project_dir.resolve())
print('Python:', sys.version.split()[0])

## 1. Full-benchmark controls

The run mode is deliberately hard-coded to `benchmark`. Batch size and PyTorch thread count remain configurable performance controls, but no environment variable can downgrade this notebook to a smoke subset.

In [ ]:
RUN_MODE = 'benchmark'
batch_size = int(os.environ.get('E5_BASELINE_BATCH_SIZE', '128'))
torch_threads = int(os.environ.get('E5_BASELINE_TORCH_THREADS', '0'))
if torch_threads > 0:
    import torch
    torch.set_num_threads(torch_threads)

config = BaselineConfig(
    run_mode=RUN_MODE,
    batch_size=batch_size,
    candidate_depth=1000,
    cache_dir='artifacts/e5_baseline_full/cache',
    artifact_dir='artifacts/e5_baseline_full',
)
if config.run_mode != 'benchmark':
    raise RuntimeError('This notebook must run in benchmark mode')
set_seed(config.seed)

print(json.dumps(config.as_dict(), indent=2, sort_keys=True))
print('Installed packages:')
print(json.dumps(package_versions(['coir-eval', 'datasets', 'faiss-cpu', 'numpy', 'pytrec-eval-terrier', 'sentence-transformers', 'torch', 'transformers']), indent=2, sort_keys=True))
print('Hardware/runtime:')
print(json.dumps(environment_metadata(config, repo_root=workspace_root), indent=2, sort_keys=True, default=str))

## 2. Load and validate the complete CosQA data

The loader reads the pinned `corpus`, `queries`, and `default/test` configurations. Titles are excluded from model inputs to follow the paper-compatible COIR text-only path.

In [ ]:
data = load_cosqa(config)
run_data = select_run_data(data, config)
if len(run_data.queries) != len(data.queries) or len(run_data.corpus) != len(data.corpus):
    raise RuntimeError('Full notebook selected a subset; refusing to produce benchmark evidence')
if len(run_data.qrels) != len(data.qrels):
    raise RuntimeError('Full notebook did not retain every test-qrels query')

print(json.dumps(data.schema, indent=2, sort_keys=True, default=str))
print(json.dumps({
    'run_mode': config.run_mode,
    'corpus_count': len(run_data.corpus),
    'query_count': len(run_data.queries),
    'qrels_query_count': len(run_data.qrels),
    'qrels_judgment_count': sum(len(rels) for rels in run_data.qrels.values()),
    'exclusions': run_data.exclusions,
}, indent=2))
print('Corpus example:', next(iter(run_data.corpus.items())))
print('Query example:', next(iter(run_data.queries.items())))
print('Qrels example:', next(iter(run_data.qrels.items())))

## 3. Run identity and cache paths

Full-run caches are stored separately from smoke-run caches. Each cache is accepted only when its metadata matches the complete configuration, code version, repository commit, and ordered source-ID fingerprint.

In [ ]:
notebook_path = project_dir / 'notebooks' / 'e5_baseline_full_experiment.ipynb'
notebook_sha256 = hashlib.sha256(notebook_path.read_bytes()).hexdigest() if notebook_path.exists() else None
identity = run_identity(config, repo_root=workspace_root)
paths = cache_paths(config, identity)
print('Run identity:', identity)
print('Notebook SHA-256:', notebook_sha256)
print(json.dumps({name: str(path) for name, path in paths.items()}, indent=2))

## 4. Encode the complete corpus with E5

E5 receives `passage: ` for corpus text. No title, answer, label, or target-document information is passed to the model.

In [ ]:
encoder = E5Encoder(config)
corpus_ids = list(run_data.corpus)
corpus_texts = [run_data.corpus[doc_id]['text'] for doc_id in corpus_ids]
corpus_metadata = expected_cache_metadata(
    config, identity=identity, kind='corpus_embeddings', ids=corpus_ids, repo_root=workspace_root
)
corpus_embeddings = load_valid_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_metadata)
if corpus_embeddings is None:
    corpus_started = time.perf_counter()
    corpus_embeddings = encoder.encode_corpus(corpus_texts)
    corpus_seconds = time.perf_counter() - corpus_started
    save_embedding_cache(paths['corpus_embeddings'], paths['corpus_metadata'], corpus_embeddings, corpus_metadata)
else:
    corpus_seconds = 0.0
print('Corpus embeddings:', corpus_embeddings.shape, 'seconds:', round(corpus_seconds, 3))

## 5. Encode the complete evaluation query set

The original query text remains the evaluation query. Query expansion is intentionally absent from this E5 baseline.

In [ ]:
query_ids = list(run_data.queries)
query_texts = [run_data.queries[query_id] for query_id in query_ids]
query_metadata = expected_cache_metadata(
    config, identity=identity, kind='query_embeddings', ids=query_ids, repo_root=workspace_root
)
query_embeddings = load_valid_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_metadata)
if query_embeddings is None:
    query_started = time.perf_counter()
    query_embeddings = encoder.encode_queries(query_texts)
    query_seconds = time.perf_counter() - query_started
    save_embedding_cache(paths['query_embeddings'], paths['query_metadata'], query_embeddings, query_metadata)
else:
    query_seconds = 0.0
print('Query embeddings:', query_embeddings.shape, 'seconds:', round(query_seconds, 3))

## 6. Exact first-stage retrieval

The ranking delegates to the official COIR exact-retrieval contract: up to 1000 candidates, COIR corpus-length ordering, cosine scoring, and COIR tie handling. HyDE and re-ranking are absent because this notebook is the E5 baseline.

In [ ]:
ranking_ids = query_ids + ['__corpus__'] + corpus_ids
ranking_metadata = expected_cache_metadata(
    config, identity=identity, kind='rankings', ids=ranking_ids, repo_root=workspace_root
)
rankings = load_valid_json_cache(paths['rankings'], paths['rankings_metadata'], ranking_metadata)
if rankings is None:
    ranking_started = time.perf_counter()
    rankings = rank_with_coir_exact(
        query_embeddings,
        corpus_embeddings,
        query_ids,
        corpus_ids,
        corpus_records=run_data.corpus,
        top_k=config.candidate_depth,
    )
    ranking_seconds = time.perf_counter() - ranking_started
    save_json_cache(paths['rankings'], paths['rankings_metadata'], rankings, ranking_metadata)
else:
    ranking_seconds = 0.0
print('Ranking queries:', len(rankings))
print('First ranking:', next(iter(rankings.items())))

## 7. Official COIR evaluation

The aggregate is computed from the full rankings and test qrels. It is never manually entered.

In [ ]:
evaluation_started = time.perf_counter()
metric = evaluate_ndcg_at_10(run_data.qrels, rankings, cutoff=10)
evaluation_seconds = time.perf_counter() - evaluation_started
print(json.dumps(metric, indent=2, sort_keys=True))

## 8. Persist the full-benchmark result and provenance

This result is marked `benchmark` only because the notebook validates that all declared test queries and corpus documents were used.

In [ ]:
environment = environment_metadata(config, repo_root=workspace_root)
result = build_result(
    config,
    data,
    run_data,
    metric,
    identity=identity,
    environment=environment,
    timings={
        'corpus_encoding': corpus_seconds,
        'query_encoding': query_seconds,
        'ranking': ranking_seconds,
        'evaluation': evaluation_seconds,
    },
    repo_root=workspace_root,
)
result['notebook_sha256'] = notebook_sha256
result['artifact_provenance']['source'] = str(notebook_path.relative_to(workspace_root)).replace('\\', '/')
result['cache_paths'] = {name: str(path) for name, path in paths.items()}
artifact_paths = write_result_artifacts(
    config,
    result,
    metadata={
        'result': result,
        'cache_metadata': {
            'corpus': corpus_metadata,
            'queries': query_metadata,
            'rankings': ranking_metadata,
        },
    },
)
print(json.dumps({
    'status': result['status'],
    'benchmark_evidence': result['benchmark_evidence'],
    'nDCG@10': result['ndcg_at_10'],
    'query_count': result['query_count'],
    'corpus_count': result['corpus_count'],
    'artifact_paths': artifact_paths,
}, indent=2))

## Interpretation boundary

Only the result written by this notebook, with `status: benchmark` and `benchmark_evidence: true`, should be compared with the paper's full CosQA E5 result. The full run can be slow on CPU; cached corpus embeddings, query embeddings, and rankings allow a later rerun to resume without recomputing completed stages.